# 10x Genomics Single Cell Data Analysis

This notebook analyzes CellRanger data with RNA-seq and TCR-seq data.

In [8]:
import scanpy as sc
import scirpy as ir
import numpy as np
import json
from pathlib import Path
import muon as mu

DATA_DIR = Path("data/C143")

In [2]:
# Load the TCR data
adata_tcr = ir.io.read_10x_vdj(DATA_DIR / "GSM4385992_C143_filtered_contig_annotations.csv.gz")

# Load the associated transcriptomics data
adata = sc.read_10x_h5(DATA_DIR / "GSM4339771_C143_filtered_feature_bc_matrix.h5")
adata.var_names_make_unique()

mdata = mu.MuData({"gex": adata, "airr": adata_tcr})
mdata

/Users/alegator1209/micromamba/envs/pytcr/lib/python3.10/site-packages/airr/schema.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream


/Users/alegator1209/micromamba/envs/pytcr/lib/python3.10/site-packages/anndata/utils.py:354: ExperimentalFeatureWarning: Support for Awkward Arrays is currently experimental. Behavior may change in the future. Please report any issues you may encounter!
  warnings.warn(msg, category, stacklevel=stacklevel)
/Users/alegator1209/micromamba/envs/pytcr/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/alegator1209/micromamba/envs/pytcr/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/alegator1209/micromamba/envs/pytcr/lib/python3.10/site-packages/mudata/_core/mudata.py:1416: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.

MuData object with n_obs × n_vars = 20859 × 33539
  2 modalities
    gex:	20857 × 33539
      var:	'gene_ids', 'feature_types', 'genome'
    airr:	2186 × 0
      uns:	'scirpy_version'
      obsm:	'airr'

In [3]:
n_cells = mdata['gex'].n_obs
n_genes = mdata['gex'].n_vars

print(f"Total cells (barcodes): {n_cells}")
print(f"Total features: {n_genes}")

Total cells (barcodes): 20857
Total features: 33539


In [ ]:
n_tcr = mdata['airr'].n_obs
print(f"Number of unique cells with TCR: {n_tcr}")

airr_cell = mdata['airr'].obs.sort_values(by='cell_id').head(1).iloc[0]
airr_cell = airr_cell.name
print(f"Name of the first cell: {airr_cell}")

Number of unique cells with TCR: 2186
Name of the first cell: AAACCTGAGCTGCAAG-1


In [ ]:
nc_gene_id = mdata['gex'].var['gene_ids'].sort_values(ascending=False).iloc[0]
print(f'Non-canonical gene id: {nc_gene_id}')

Non-canonical gene id: nCoV


In [ ]:
fst_gene_name = mdata['gex'].var.index.sort_values(ascending=True)[0]
print(f'First gene name: {fst_gene_name}')

First gene name: A1BG


In [7]:
# Create output JSON
output = {
    "n_cells": n_cells,
    "n_tcr": n_tcr,
    "nc_gene_id": nc_gene_id,
    "fst_gene_id": fst_gene_name,
    "airr_cell": airr_cell
}
print("Output JSON:")
print(json.dumps(output, indent=2))

# Save to file
# with open('output.json', 'w') as f:
#     json.dump(output, f, indent=2)
# print("\nSaved to output.json")

Output JSON:
{
  "n_cells": 20857,
  "n_tcr": 2186,
  "nc_gene_id": "nCoV",
  "fst_gene_id": "A1BG",
  "airr_cell": "AAACCTGAGCTGCAAG-1"
}
